# Practical Verification — Rootkit Detection via Kernel Timing
## العمل التطبيقي — كشف الـ Rootkits عبر تحليل توقيتات الـ Kernel

**Reference Paper:** Landauer et al. (2025), *Trace of the Times*, ACM DTRAP 6(4), 1-26

**Authors:** Ahmed Atiyah AL-Zahrani (447001014) & Mohammed Salem Alghamdi (447000003) — CYBS60201 OS Security — Al-Baha University

---

### How to use this notebook / كيف تستخدم هذا الـ notebook

**EN:** Run each cell with `Shift+Enter`. Figures appear inline. Take screenshots for your presentation.

**AR:** شغّل كل خلية بـ `Shift+Enter`. الأشكال راح تظهر تحت كل خلية. خذ سكرين شوت للعرض.


## Cell 1 — Imports and setup / الاستيرادات والإعدادات

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

# Reproducibility — seed = student ID / تثبيت العشوائية
SEED = 447001014
rng = np.random.default_rng(SEED)

print(f"Setup complete. Random seed: {SEED}")

## Cell 2 — Define kernel functions and scenarios / تعريف دوال الـ kernel والسيناريوهات

The paper probes 4 functions inside the `getdents` syscall.
CARAXES rootkit wraps `filldir64` and injects extra code (~430 ns delay).

الورقة تراقب 4 دوال داخل `getdents`. الـ rootkit يلتف حول `filldir64` ويضيف ~430 نانوثانية.

In [ ]:
FUNCTIONS = {
    "iterate_dir":        {"base": 2600, "spread": 480},
    "filldir64":          {"base": 1850, "spread": 360},   # rootkit target
    "verify_dirent_name": {"base": 1380, "spread": 240},
    "touch_atime":        {"base":  920, "spread": 210},
}

SCENARIOS = ["default", "file_count", "system_load", "ls_basic", "filename_length"]

SCEN_NOISE = {
    "default": 1.00, "file_count": 1.15, "system_load": 1.55,
    "ls_basic": 1.08, "filename_length": 1.22,
}

ROOTKIT_TARGET = "filldir64"
ROOTKIT_SHIFT_NS = 430
EVENTS_PER_BATCH = 100
QUANTILES = np.linspace(0, 1 - 1/10, 10)[1:]   # 9 quantiles

print(f"Functions probed: {list(FUNCTIONS.keys())}")
print(f"Scenarios: {SCENARIOS}")
print(f"Rootkit target: {ROOTKIT_TARGET} (adds {ROOTKIT_SHIFT_NS} ns)")
print(f"Quantiles used: {len(QUANTILES)} quantiles")

## Cell 3 — Generate synthetic data / توليد البيانات التركيبية

Produces 750 normal batches + 500 rootkit batches (matches paper's setup).

ينتج 750 دفعة طبيعية + 500 دفعة مع rootkit (نفس إعداد الورقة).

In [ ]:
def make_batch(scenario, infected):
    """Generate one batch of delta time measurements."""
    noise = SCEN_NOISE[scenario]
    batch = {}
    for fname, p in FUNCTIONS.items():
        n = EVENTS_PER_BATCH
        # Multimodal distribution: ~70% fast path, ~30% slow path
        n_slow = rng.binomial(n, 0.30)
        n_fast = n - n_slow
        fast = rng.gamma(shape=4.0, scale=p["spread"] * noise / 4.0, size=n_fast) + p["base"]
        slow = rng.gamma(shape=3.0, scale=p["spread"] * noise / 2.0, size=n_slow) + p["base"] * 1.6
        deltas = np.concatenate([fast, slow])
        if infected and fname == ROOTKIT_TARGET:
            deltas = deltas + ROOTKIT_SHIFT_NS + rng.normal(0, 55, size=deltas.size)
        batch[fname] = np.clip(deltas, 1, None)
    return batch

normal_batches, rootkit_batches = [], []
for scen in SCENARIOS:
    for _ in range(150):
        normal_batches.append((scen, make_batch(scen, infected=False)))
    for _ in range(100):
        rootkit_batches.append((scen, make_batch(scen, infected=True)))

print(f"Normal batches generated:  {len(normal_batches)}")
print(f"Rootkit batches generated: {len(rootkit_batches)}")
print(f"Total batches: {len(normal_batches) + len(rootkit_batches)}")

## Cell 4 — Detection algorithm / خوارزمية الكشف

Three functions:
1. `quantile_vector` — compute 9 quantiles per function
2. `fit_baseline` — learn mean + inverse covariance from clean data
3. `batch_pvalue` — score a test batch using Mahalanobis distance and chi-square

In [ ]:
def quantile_vector(batch):
    """Compute 9 quantiles per function."""
    return {f: np.quantile(d, QUANTILES) for f, d in batch.items()}

def fit_baseline(train_batches):
    """Fit clean-data baseline: mean vector + inverse covariance per function."""
    stacks = {f: [] for f in FUNCTIONS}
    for _, b in train_batches:
        qv = quantile_vector(b)
        for f in FUNCTIONS:
            stacks[f].append(qv[f])
    mean, cov_inv = {}, {}
    for f in FUNCTIONS:
        arr = np.array(stacks[f])
        mean[f] = arr.mean(axis=0)
        cov = np.cov(arr, rowvar=False)
        cov_inv[f] = np.linalg.pinv(cov)
    return mean, cov_inv

def batch_pvalue(batch, mean, cov_inv):
    """Score one batch: lowest chi-square p-value across all functions."""
    qv = quantile_vector(batch)
    pvals = []
    for f in FUNCTIONS:
        diff = qv[f] - mean[f]
        mhd_sq = float(diff @ cov_inv[f] @ diff)
        p = 1.0 - chi2.cdf(mhd_sq, df=len(QUANTILES))
        pvals.append(p)
    return min(pvals)

print("Detection functions defined.")

## Cell 5 — Train detector and score all test batches / تدريب الكاشف وتسجيل كل دفعات الاختبار

In [ ]:
# Semi-supervised split: 1/3 of normal data for training, rest for testing
rng.shuffle(normal_batches)
n_train = len(normal_batches) // 3
train_batches = normal_batches[:n_train]
test_normal = normal_batches[n_train:]
test_rootkit = rootkit_batches

mean, cov_inv = fit_baseline(train_batches)

p_normal = np.array([batch_pvalue(b, mean, cov_inv) for _, b in test_normal])
p_rootkit = np.array([batch_pvalue(b, mean, cov_inv) for _, b in test_rootkit])

print(f"Trained on:    {len(train_batches)} clean batches")
print(f"Test normal:   {len(test_normal)} batches  (lowest p-value: {p_normal.min():.2e})")
print(f"Test rootkit:  {len(test_rootkit)} batches  (highest p-value: {p_rootkit.max():.2e})")

## Cell 6 — Find optimal threshold and compute metrics / إيجاد أفضل threshold وحساب المقاييس

In [ ]:
best = None
for thr in np.logspace(-30, -1, 200):
    tp = int(np.sum(p_rootkit < thr))
    fn = len(p_rootkit) - tp
    fp = int(np.sum(p_normal < thr))
    tn = len(p_normal) - fp
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    if best is None or f1 > best["f1"]:
        best = dict(
            threshold=thr, tp=tp, fp=fp, tn=tn, fn=fn,
            precision=prec, recall=rec, f1=f1,
            accuracy=(tp + tn) / (tp + tn + fp + fn),
            false_positive_rate=fp / (fp + tn) if (fp + tn) else 0,
        )

print("=" * 70)
print("  DETECTION RESULTS")
print("=" * 70)
print(f"  Optimal threshold        : {best['threshold']:.3e}")
print(f"  True positives  (TP)     : {best['tp']}")
print(f"  False positives (FP)     : {best['fp']}")
print(f"  True negatives  (TN)     : {best['tn']}")
print(f"  False negatives (FN)     : {best['fn']}")
print(f"  Precision                : {best['precision']:.4f}")
print(f"  Recall                   : {best['recall']:.4f}")
print(f"  F1 score                 : {best['f1']:.4f}")
print(f"  Accuracy                 : {best['accuracy']:.4f}")
print(f"  False positive rate      : {best['false_positive_rate']:.4f}")
print("=" * 70)
print(f"  Paper reports F1 = 0.99  (TP=499, FN=1, FP=9)")
print("=" * 70)

# Save / احفظ
with open("practical_results.json", "w", encoding="utf-8") as fh:
    serializable = {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                    for k, v in best.items()}
    serializable["seed"] = SEED
    json.dump(serializable, fh, indent=2)
print("\nSaved: practical_results.json")

## Cell 7 — Figure 2: timing distribution shift / الشكل 2: انزياح توزيع التوقيت

In [ ]:
norm_target = np.concatenate([b[ROOTKIT_TARGET] for _, b in test_normal[:60]])
rk_target = np.concatenate([b[ROOTKIT_TARGET] for _, b in test_rootkit[:60]])

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
bins = np.linspace(1000, 5200, 70)
ax.hist(norm_target, bins=bins, color="#2E6B43", alpha=0.62,
        label="Normal (no rootkit)", edgecolor="white", linewidth=0.3)
ax.hist(rk_target, bins=bins, color="#B23A3A", alpha=0.62,
        label="Rootkit active (CARAXES)", edgecolor="white", linewidth=0.3)
ax.axvline(np.median(norm_target), color="#1F4A2E", linestyle="--", linewidth=1.4)
ax.axvline(np.median(rk_target), color="#7A1F1F", linestyle="--", linewidth=1.4)
shift = np.median(rk_target) - np.median(norm_target)
ymax = ax.get_ylim()[1]
ax.set_ylim(0, ymax * 1.18)
ax.annotate("", xy=(np.median(rk_target), ymax * 0.70),
            xytext=(np.median(norm_target), ymax * 0.70),
            arrowprops=dict(arrowstyle="<->", color="#333333", lw=1.3))
ax.text((np.median(norm_target) + np.median(rk_target)) / 2, ymax * 0.78,
        f"median shift  +{shift:.0f} ns", ha="center", fontsize=10,
        color="#222222", fontweight="bold")
ax.set_xlabel("filldir64 execution delta time (nanoseconds)")
ax.set_ylabel("Frequency")
ax.set_title("Delta Time Distribution of the Rootkit-Wrapped Kernel Function",
             fontweight="bold")
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig2_distribution_shift.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Median shift detected: +{shift:.0f} ns")

## Cell 8 — Figure 3: anomaly score separation / الشكل 3: فصل درجات الكشف

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
eps = 1e-31
pn = np.clip(p_normal, eps, 1)
pr = np.clip(p_rootkit, eps, 1)
log_bins = np.logspace(-31, 0, 55)
ax.hist(pn, bins=log_bins, color="#2E6B43", alpha=0.62,
        label="Normal batches", edgecolor="white", linewidth=0.3)
ax.hist(pr, bins=log_bins, color="#B23A3A", alpha=0.62,
        label="Rootkit batches", edgecolor="white", linewidth=0.3)
ax.axvline(best["threshold"], color="#222222", linestyle="--", linewidth=1.6,
           label="Decision threshold")
ax.set_xscale("log")
ax.set_xlabel("Chi-square p-value  (batch anomaly score, log scale)")
ax.set_ylabel("Number of batches")
ax.set_title("Anomaly Score Separation Between Normal and Rootkit Batches",
             fontweight="bold")
ax.legend(loc="upper center")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig3_score_separation.png", dpi=200, bbox_inches="tight")
plt.show()

## Done / تم

**Output files / الملفات المُولّدة:**
- `practical_results.json` — detection metrics
- `fig2_distribution_shift.png` — high-res figure for the report
- `fig3_score_separation.png` — high-res figure for the report

**Take screenshots / خذ سكرين شوت:**
- Right-click on each figure in VS Code → Save Image As
- Or use Windows: `Win + Shift + S`